# GeoMM-Bench — Unified Experiment

One backbone (`openai/clip-vit-base-patch32`) for all CLIP approaches; Grounding DINO and BLIP-2 are separate, non-CLIP models. This notebook calls the same package code as `run_geomm_bench.py` and writes the same results file.

**Finding:** only text classifies reasonably (~0.73 macro-F1); vision, grounding, VQA and added-modality fusion all fail or are unreliable, and adding Full Wave Sonic makes it worse. *Backbone caveat:* vision-CLIP is backbone-sensitive (OpenAI CLIP ~0.09 vs open-clip/LAION ~0.62), so the claim is "no off-the-shelf approach is reliable," not that vision is uniformly near-random.

## 1. Setup

In [ ]:
# Dependencies (uncomment on first run)
# %pip install -q torch transformers Pillow numpy matplotlib pdf2image
# macOS: brew install poppler   |   Linux: apt-get install poppler-utils

In [ ]:
import os, sys, json
from collections import Counter
sys.path.insert(0, os.path.abspath('.'))
from geomm_bench import baselines as B
print('backbone:', B.CLIP_MODEL_NAME)

## 2. Ground truth (11 labelled intervals)

In [ ]:
GT = json.load(open('data/ground_truth.json'))['intervals']
print(len(GT), 'intervals;', dict(Counter(g['lithology'] for g in GT)))

## 3. Source PDFs (optional — required for the image approaches)

Set these to your operator rasters; leave them missing to run text-only only. The PDFs are not redistributed (see `DATASHEET.md`).

In [ ]:
LOGS_PDF = 'vilkyciai22_logs500.pdf'    # set to your path
FWS_PDF  = 'vilkyciai22_fws_im_dt.pdf'  # set to your path
LOGS_PDF = LOGS_PDF if os.path.exists(LOGS_PDF) else None
FWS_PDF  = FWS_PDF  if os.path.exists(FWS_PDF)  else None
print('logs:', LOGS_PDF, '| fws:', FWS_PDF)

## 4. Run the benchmark (single backbone, all approaches)

In [ ]:
import run_geomm_bench as R

results = R.run(R.ALL_APPROACHES, GT, LOGS_PDF, FWS_PDF)
doc = {
    'benchmark': 'GeoMM-Bench', 'version': '0.2.0', 'well': 'Vilkyciai-22',
    'n_intervals': len(GT), 'backbone': R.BACKBONE,
    'metric': 'macro_f1_present_classes', 'results': results,
    'baselines': {'random_four_class_f1': 0.25}, 'notes': [R.CAVEAT],
}
OUT_JSON = 'results/geomm_bench_results.json'
os.makedirs('results', exist_ok=True)
json.dump(doc, open(OUT_JSON, 'w'), indent=2)
for a in R.ALL_APPROACHES:
    print(f"{a:24s} F1={results[a]['macro_f1']} acc={results[a]['accuracy']}")

## 5. Figure (plotted from the results file, so it cannot drift)

In [ ]:
import subprocess
subprocess.run([sys.executable, 'scripts/make_results_figure.py',
                '--results', OUT_JSON,
                '--out', 'images/Figure2_Results_Comparison.png'])

## 6. Conclusion (written by hand from the numbers above)

Only text-only classification succeeds (~0.73 macro-F1). Every off-the-shelf vision, grounding, VQA, multimodal and added-modality approach fails or is unreliable on well-log display interpretation, and adding Full Wave Sonic degrades performance.

**Backbone caveat:** vision-CLIP is backbone-sensitive (OpenAI CLIP ~0.09 vs open-clip/LAION ~0.62 macro-F1), so the honest claim is that *no off-the-shelf approach is reliable* — not that vision is uniformly near-random. The limitation is the visual representation, motivating a domain-adapted encoder rather than more data or bigger off-the-shelf models.